In [7]:
#Challenge 1
import pandas as pd
data = {
    "age": [20, 21, None, 22, 23, None, 24, 25],
    "cgpa": [8.2, None, 7.5, 8.8, None, 7.1, 9.0, None],
    "attendance": [90, 85, None, 92, 78, None, 95, 88],
    "projects": [2, 3, 1, None, 4, 2, None, 5]
}

df = pd.DataFrame(data)
print('Missing Values Per Column:')
for col in df.columns:
    print(col+':' , df[col].isna().sum())
print(f'Total Missing Values: {df.isna().sum().sum()}')

print('Missing Values Percentage Per Column:')
for col in df.columns:
    print(f'{col}: {df[col].isna().mean()*100:.2f}%')

Missing Values Per Column:
age: 2
cgpa: 3
attendance: 2
projects: 2
Total Missing Values: 9
Missing Values Percentage Per Column:
age: 25.00%
cgpa: 37.50%
attendance: 25.00%
projects: 25.00%


In [18]:
#Challenge 2
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
imp_mean = SimpleImputer(strategy='mean')
imp_median = SimpleImputer(strategy='median')
exp_a = make_column_transformer(
    (imp_mean , ['age','cgpa']),
    (imp_median , ['attendance','projects']),
    remainder = 'passthrough')
print(exp_a.fit_transform(df))

exp_b = imp_median
print(exp_b.fit_transform(df))

'''Median might be prefered for cgpa and attendance because people with 
9 or above gpa will tend to pull up the mean same goes for attendance , 
perfect attendance pulls the mean up'''


[[20.    8.2  90.    2.  ]
 [21.    8.12 85.    3.  ]
 [22.5   7.5  89.    1.  ]
 [22.    8.8  92.    2.5 ]
 [23.    8.12 78.    4.  ]
 [22.5   7.1  89.    2.  ]
 [24.    9.   95.    2.5 ]
 [25.    8.12 88.    5.  ]]
[[20.   8.2 90.   2. ]
 [21.   8.2 85.   3. ]
 [22.5  7.5 89.   1. ]
 [22.   8.8 92.   2.5]
 [23.   8.2 78.   4. ]
 [22.5  7.1 89.   2. ]
 [24.   9.  95.   2.5]
 [25.   8.2 88.   5. ]]


'Median might be prefered for cgpa and attendance because people with \n9 or above gpa will tend to pull up the mean same goes for attendance , \nperfect attendance pulls the mean up'

In [44]:
#Challenge 3 and 4 Combined
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from sklearn.pipeline import make_pipeline
data = {
    "hours_studied": [2, 3, 4, 5, 6, 7, 8, 9, None, 10, 11, None],
    "attendance": [60, 65, 70, None, 75, 80, None, 85, 90, 92, None, 95],
    "projects": [0, 1, 1, 2, 2, None, 3, 3, 4, None, 4, 5],
    "placement": [
        "No", "No", "No", "No",
        "Yes", "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes", "Yes"
    ]
}
df = pd.DataFrame(data)
imp_median = SimpleImputer(strategy='median')
#imp_mostf = Simple_Imputer(strategy='most_frequent')
X = df[["hours_studied","attendance","projects"]]
y = [1 if x == 'Yes' else 0 for x in df['placement']]
mnb = MultinomialNB()

#Method A(Bad WorkFlow)
X_transformed = imp_median.fit_transform(X)
X_train , X_test , y_train , y_test = train_test_split(X_transformed , y , test_size=0.3 , random_state=6)

mnb.fit(X_train , y_train)
train_pred = mnb.predict(X_train)
test_pred = mnb.predict(X_test)
print('for The method A(Bad WorkFlow):')
print('Training Accuracy:',metrics.accuracy_score(y_train , train_pred))
print('Testing Accuracy:',metrics.accuracy_score(y_test , test_pred))

#Method B(good WorkFlow)
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.3 , random_state=5)
pipe = make_pipeline(imp_median , mnb)
pipe.fit(X_train , y_train)
train_pred = pipe.predict(X_train)
test_pred = pipe.predict(X_test)
print('for The method B(Good WorkFlow):')
print('Training Accuracy:',metrics.accuracy_score(y_train , train_pred))
print('Testing Accuracy:',metrics.accuracy_score(y_test , test_pred))

for The method A(Bad WorkFlow):
Training Accuracy: 1.0
Testing Accuracy: 1.0
for The method B(Good WorkFlow):
Training Accuracy: 0.875
Testing Accuracy: 1.0


In [73]:
#Challenge 5
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
student_data = {
    'age': [23, None, 22, 24, 24, 21, None, 22, 22, 24, 23, 22, 24, 21, None, 21, 23, 24, 20, 23, None, 24, 23, 20, 20, 22, 22, 21, None, 23, 22, 23, 23, 20, None, 24, 22, 24, 20, 21, 23, None, 23, 21, 21, 20, 21, 24, None, 23], 
    'cgpa': [6.92, 8.6, 8.32, 9.17, 6.66, 7.49, 6.69, 8.87, None, None, 8.16, 6.12, None, 7.71, None, 9.52, None, 7.24, 8.17, 7.98, 9.65, 9.21, 8.84, 8.05, 8.23, 9.67, None, 7.05, 7.13, 6.63, 6.06, 7.61, 7.5, 7.12, 6.05, 6.76, 8.7, 9.0, 8.3, 9.52, 8.47, 9.48, None, 7.71, 6.36, 7.41, 8.54, 8.53, None, 7.04], 
    'attendance': [99.0, None, 99.0, 78.0, 67.0, None, 69.0, 90.0, 78.0, 91.0, 73.0, 79.0, 79.0, None, 77.0, 96.0, 96.0, 68.0, None, None, 79.0, 93.0, 77.0, None, 71.0, 86.0, 92.0, 66.0, 70.0, None, 92.0, 84.0, 94.0, None, 92.0, 89.0, 97.0, 65.0, 91.0, 77.0, 67.0, 70.0, 72.0, 91.0, 73.0, 97.0, 88.0, 79.0, 96.0, 96.0], 
    'department': ['Computer Science', 'Mechanical', 'Computer Science', 'Computer Science', 'Information Technology', 'Mechanical', 'Mechanical', 'Information Technology', 'Electronics', 'Computer Science', 'Civil', 'Computer Science', 'Computer Science', 'Electronics', 'Computer Science', 'Information Technology', 'Information Technology', 'Mechanical', 'Civil', 'Computer Science', 'Computer Science', 'Electronics', 'Information Technology', 'Civil', 'Mechanical', 'Information Technology', 'Mechanical', 'Electronics', 'Electronics', 'Computer Science', 'Civil', 'Mechanical', 'Information Technology', 'Electronics', 'Computer Science', 'Computer Science', 'Mechanical', 'Electronics', 'Civil', 'Electronics', 'Mechanical', 'Mechanical', 'Electronics', 'Mechanical', 'Electronics', 'Information Technology', 'Electronics', 'Electronics', 'Mechanical', 'Mechanical'], 
    'projects': [0, 0, 1, None, 2, 3, 0, 0, 1, 1, None, 3, 1, 0, 3, 3, 0, None, 0, 3, 4, 4, 2, 0, None, 2, 2, 2, 3, 0, None, 2, 0, 3, 3, 2, 0, None, 0, 4, 1, 1, 1, 2, None, 0, 3, 0, 3, 0], 
    'placement': ['Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes']
}
df = pd.DataFrame(student_data)
X = df[['age','cgpa','attendance','department','projects']]
y=[1 if x=='Yes' else 0 for x in df['placement']]

num_cols = ['age','cgpa','attendance','projects']
categ_cols = ['department']
Imputer = SimpleImputer(strategy='median')

numeric_pipe = make_pipeline(Imputer,StandardScaler())
col_transformer = ColumnTransformer([
    ('num' , numeric_pipe , num_cols),
    ('cat' , OneHotEncoder(handle_unknown='ignore'), categ_cols)])

logreg = LogisticRegression(max_iter=500)

pipe = make_pipeline(col_transformer , logreg)
scores = cross_val_score(pipe , X , y , cv=10, scoring='accuracy')
print('Cross Val Score:',scores.mean())

Cross Val Score: 0.74
